# 03 — Collaborative Filtering (ALS)

"People who listened to what you listened to also played this": Alternating Least
Squares matrix factorization over the LastFM-1K implicit-feedback play data,
using the `implicit` library (CPU-friendly — this machine has no GPU).

Evaluated with a **temporal per-user split** (train on earlier plays, hold out each
user's most recent plays) — a random split would leak future listens into training.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd

from src.splitting import temporal_split, held_out_as_dict
from src.collaborative import ALSRecommender
from src.evaluation import evaluate_model

# Train on the FULL interaction history (all 740k tracks), not just the ~1.3% that
# happen to have a Spotify audio-feature match. ALS doesn't need audio features to
# learn collaborative patterns, and restricting it to the matched subset would throw
# away 91% of the real signal for no reason. Items are keyed by `join_key`
# (artist+track, normalized) since that's the only identifier both datasets share;
# audio features get attached later, per-recommendation, only where a match exists.
interactions = pd.read_parquet('../data/raw/interactions/lastfm_1k_train.parquet')
print(f'{len(interactions):,} full interaction rows loaded (all tracks, not just audio-feature-matched)')

13,365,803 full interaction rows loaded (all tracks, not just audio-feature-matched)


In [2]:
train, test = temporal_split(
    interactions, user_col='user_id', item_col='join_key', timestamp_col='timestamp',
    test_frac=0.2, min_events_per_user=5,
)
print(f'train: {len(train):,} events, {train["user_id"].nunique():,} users')
print(f'test:  {len(test):,} events, {test["user_id"].nunique():,} users')
held_out = held_out_as_dict(test, 'user_id', 'join_key')

train: 10,692,949 events, 792 users
test:  2,672,850 events, 792 users


In [3]:
als = ALSRecommender(factors=64, regularization=0.05, iterations=20)
als.fit(train, user_col='user_id', item_col='join_key')
print(f'fit: {len(als.user_to_idx):,} users x {len(als.item_to_idx):,} items')

C:\Users\User\Desktop\MUSIC-RECOMMENDER\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

fit: 792 users x 657,551 items


## Baseline: recommend the most popular tracks overall (the floor everything else must beat)

In [4]:
popularity_ranking = train['join_key'].value_counts().index.tolist()

def popularity_recommend(user_id, k=10):
    return popularity_ranking[:k]

def als_recommend(user_id, k=10):
    return als.recommend(user_id, k=k)

test_users = list(held_out.keys())
pop_result = evaluate_model(popularity_recommend, test_users, held_out, k=10)
als_result = evaluate_model(als_recommend, test_users, held_out, k=10)

print('Popularity baseline:', {k: v for k, v in pop_result.items() if k != 'all_recommendations'})
print('ALS collaborative:  ', {k: v for k, v in als_result.items() if k != 'all_recommendations'})

Popularity baseline: {'precision_at_k': 0.09646464646464648, 'recall_at_k': 0.0010615215019801434, 'n_users_evaluated': 792}
ALS collaborative:   {'precision_at_k': 0.0880050505050505, 'recall_at_k': 0.0012618494500096827, 'n_users_evaluated': 792}


If ALS doesn't clearly beat the popularity baseline, that's a real finding to report
honestly in the case study, not something to paper over — it would mean the interaction
signal here is too sparse for personalization to add value over "just recommend what's
popular," which is itself informative about the limits of this dataset.

In [5]:
import json
results = {
    'popularity_baseline': {k: v for k, v in pop_result.items() if k != 'all_recommendations'},
    'als_collaborative': {k: v for k, v in als_result.items() if k != 'all_recommendations'},
}
with open('../reports/figures/03_collaborative_results.json', 'w') as f:
    json.dump(results, f, indent=2)

import pickle
with open('../data/raw/interactions/als_model.pkl', 'wb') as f:
    pickle.dump(als, f)
with open('../data/raw/interactions/held_out.pkl', 'wb') as f:
    pickle.dump(held_out, f)
# Saved for notebook 04, so the popularity baseline there stays consistent with this
# one (full 740k-track catalog, join_key space) rather than being silently recomputed
# over a different (smaller, Spotify-matched-only) subset.
with open('../data/raw/interactions/popularity_ranking.pkl', 'wb') as f:
    pickle.dump(popularity_ranking, f)
with open('../data/raw/interactions/catalog_size.pkl', 'wb') as f:
    pickle.dump(len(als.item_to_idx), f)
print('saved ALS model + eval results for notebook 04')

saved ALS model + eval results for notebook 04
